1. 调用摄像头拍一张照片

In [ ]:
import cv2
from IPython.display import display, Image

camera = cv2.VideoCapture(0)
camera.set(cv2.CAP_PROP_FOURCC,cv2.VideoWriter_fourcc('M','J','P','G'))
if not camera.isOpened():
    raise IOError("Impossible d'ouvrir la webcam")
ret, frame = camera.read()
if not ret:
    raise IOError("Impossible de capturer une image")
display(Image(data=cv2.imencode('.jpg', frame)[1]))
camera.release()


2. 实时查看usb摄像头画面

In [ ]:
import cv2
import numpy as np
from IPython.display import display, clear_output,Image

# Initialize the camera
camera = cv2.VideoCapture(0)  # Use 0 for the default camera
# Set the codec to MJPG if it is supported
if camera.isOpened():
    camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc('M', 'J', 'P', 'G'))
else:
    raise IOError("Cannot open the webcam")
try:
    while True:
        # Capture frame-by-frame
        ret, frame = camera.read()
        if not ret:
            raise IOError("Cannot capture frame")
        # Display the image
        clear_output(wait=True)
        # Afficher l'image capturée
        display(Image(data=cv2.imencode('.jpg', frame)[1]))
finally:
    # When everything done, release the capture
    camera.release()

3. 实时查看小车画面（小车需运行remote_control.py或任何将视频流传输到ip:9000/mjpg的程序）

In [ ]:
import cv2
import numpy as np
from IPython.display import display, clear_output,Image

# Initialize the camera
url = "http://10.129.137.156:9000/mjpg"
camera = cv2.VideoCapture(url)
# Set the codec to MJPG if it is supported
if camera.isOpened():
    camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc('M', 'J', 'P', 'G'))
else:
    raise IOError("Cannot open the webcam")
try:
    while True:
        # Capture frame-by-frame
        ret, frame = camera.read()
        if not ret:
            raise IOError("Cannot capture frame")
        # Display the image
        clear_output(wait=True)
        # Afficher l'image capturée
        display(Image(data=cv2.imencode('.jpg', frame)[1]))
finally:
    # When everything done, release the capture
    camera.release()

4. 使用torch进行yolo识别usb摄像头画面中的物体

In [ ]:
import cv2
import numpy as np
from IPython.display import display, clear_output,Image
from ultralytics import YOLO
from time import time
# Load a model
model = YOLO('yolov8n.pt')  # pretrained YOLOv8n model
# Initialize the camera
camera = cv2.VideoCapture(0)  # Use 0 for the default camera
# Set the codec to MJPG if it is supported
if camera.isOpened():
    # camera.set(cv2.CAP_PROP_FRAME_WIDTH, 1280.0)
    # camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 720.0)
    camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc('M', 'J', 'P', 'G'))
else:
    raise IOError("Cannot open the webcam")
try:
    while True:
        # Capture frame-by-frame
        ret, frame = camera.read()
        if not ret:
            raise IOError("Cannot capture frame")
        s = time()
        results = model(frame,conf=0.25,iou=0.5,verbose=False)
        print(time()-s)
        for r in results:
            im = r.plot()
        # Display the image
        clear_output(wait=True)
        # Afficher l'image capturée
        display(Image(data=cv2.imencode('.jpg', im)[1]))
finally:
    # When everything done, release the capture
    camera.release()

5. 使用torch进行yolo识别摄像头画面中的物体（小车需运行remote_control.py或任何将视频流传输到ip:9000/mjpg的程序）

In [ ]:
import cv2
import numpy as np
from IPython.display import display, clear_output,Image
from ultralytics import YOLO
from time import time
# Load a model
model = YOLO('yolov8n.pt')  # pretrained YOLOv8n model
# Initialize the camera
url = "http://10.129.137.156:9000/mjpg"
camera = cv2.VideoCapture(url)
# Set the codec to MJPG if it is supported
if camera.isOpened():
    # camera.set(cv2.CAP_PROP_FRAME_WIDTH, 1280.0)
    # camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 720.0)
    camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc('M', 'J', 'P', 'G'))
else:
    raise IOError("Cannot open the webcam")
try:
    while True:
        # Capture frame-by-frame
        ret, frame = camera.read()
        if not ret:
            raise IOError("Cannot capture frame")
        s = time()
        results = model(frame,conf=0.25,iou=0.5,verbose=False)
        print(time()-s)
        for r in results:
            im = r.plot()
        # Display the image
        clear_output(wait=True)
        # Afficher l'image capturée
        display(Image(data=cv2.imencode('.jpg', im)[1]))
finally:
    # When everything done, release the capture
    camera.release()

In [ ]:
6. 使用华为昇腾NPU进行yolo识别usb摄像头画面中的物体

In [ ]:
import os

# Verify the path
print(os.environ['LD_LIBRARY_PATH'])
import cv2
import numpy as np
from IPython.display import display, clear_output,Image
from time import time
from ais_bench.infer.interface import InferSession
from ultralytics.utils import ASSETS, YAML
from ultralytics.utils.checks import check_yaml

CLASSES = YAML.load(check_yaml('coco128.yaml'))['names']

colors = np.random.uniform(0, 255, size=(len(CLASSES), 3))

model = InferSession(device_id=0, model_path="yolov8n.om")

def draw_bounding_box(img, class_id, confidence, x, y, x_plus_w, y_plus_h):
    label = f'{CLASSES[class_id]} ({confidence:.2f})'
    color = colors[class_id]
    cv2.rectangle(img, (x, y), (x_plus_w, y_plus_h), color, 2)
    cv2.putText(img, label, (x - 10, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)


def main(original_image):
    [height, width, _] = original_image.shape
    length = max((height, width))
    image = np.zeros((length, length, 3), np.uint8)
    image[0:height, 0:width] = original_image
    scale = length / 640

    blob = cv2.dnn.blobFromImage(image, scalefactor=1 / 255, size=(640, 640), swapRB=True)
    
    begin_time = time()
    outputs = model.infer(feeds=blob, mode="static")
    end_time = time()
    print("om infer time:", end_time - begin_time)

    outputs = np.array([cv2.transpose(outputs[0][0])])
    rows = outputs.shape[1]

    boxes = []
    scores = []
    class_ids = []

    for i in range(rows):
        classes_scores = outputs[0][i][4:]
        (minScore, maxScore, minClassLoc, (x, maxClassIndex)) = cv2.minMaxLoc(classes_scores)
        if maxScore >= 0.25:
            box = [
                outputs[0][i][0] - (0.5 * outputs[0][i][2]), outputs[0][i][1] - (0.5 * outputs[0][i][3]),
                outputs[0][i][2], outputs[0][i][3]]
            boxes.append(box)
            scores.append(maxScore)
            class_ids.append(maxClassIndex)

    result_boxes = cv2.dnn.NMSBoxes(boxes, scores, 0.25, 0.45, 0.5)

    detections = []
    for i in range(len(result_boxes)):
        index = result_boxes[i]
        box = boxes[index]
        detection = {
            'class_id': class_ids[index],
            'class_name': CLASSES[class_ids[index]],
            'confidence': scores[index],
            'box': box,
            'scale': scale}
        detections.append(detection)
        draw_bounding_box(original_image, class_ids[index], scores[index], round(box[0] * scale), round(box[1] * scale),
                          round((box[0] + box[2]) * scale), round((box[1] + box[3]) * scale))

# Initialize the camera
camera = cv2.VideoCapture(0)  # Use 0 for the default camera

# Set the codec to MJPG if it is supported
if camera.isOpened():
    # camera.set(cv2.CAP_PROP_FRAME_WIDTH, 1280.0)
    # camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 720.0)
    camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc('M', 'J', 'P', 'G'))
else:
    raise IOError("Cannot open the webcam")

    # Define the codec and create VideoWriter object
# Get the width and height of the frames
frame_width = int(camera.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Frame width: {frame_width}, Frame height: {frame_height}")

# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi', fourcc, 30.0, (frame_width, frame_height))  # 20.0 is the frame rate

try:
    _start_time = time()
    while time() - _start_time >= 0:
        # Capture frame-by-frame
        ret, frame = camera.read()
        if not ret:
            raise IOError("Cannot capture frame")
        main(frame)
        out.write(frame)

        # Display the image
        clear_output(wait=True)
        
        # Afficher l'image capturée
        display(Image(data=cv2.imencode('.jpg', frame)[1]))

finally:
    # When everything done, release the capture
    camera.release()
    out.release()

7. 使用华为昇腾NPU进行yolo识别摄像头画面中的物体（小车需运行remote_control.py或任何将视频流传输到ip:9000/mjpg的程序）

In [ ]:
import os

# Verify the path
print(os.environ['LD_LIBRARY_PATH'])
import cv2
import numpy as np
from IPython.display import display, clear_output,Image
from time import time
from ais_bench.infer.interface import InferSession
from ultralytics.utils import ASSETS, YAML
from ultralytics.utils.checks import check_yaml

CLASSES = YAML.load(check_yaml('coco128.yaml'))['names']

colors = np.random.uniform(0, 255, size=(len(CLASSES), 3))

model = InferSession(device_id=0, model_path="yolov8n.om")

def draw_bounding_box(img, class_id, confidence, x, y, x_plus_w, y_plus_h):
    label = f'{CLASSES[class_id]} ({confidence:.2f})'
    color = colors[class_id]
    cv2.rectangle(img, (x, y), (x_plus_w, y_plus_h), color, 2)
    cv2.putText(img, label, (x - 10, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)


def main(original_image):
    [height, width, _] = original_image.shape
    length = max((height, width))
    image = np.zeros((length, length, 3), np.uint8)
    image[0:height, 0:width] = original_image
    scale = length / 640

    blob = cv2.dnn.blobFromImage(image, scalefactor=1 / 255, size=(640, 640), swapRB=True)
    
    begin_time = time()
    outputs = model.infer(feeds=blob, mode="static")
    end_time = time()
    print("om infer time:", end_time - begin_time)

    outputs = np.array([cv2.transpose(outputs[0][0])])
    rows = outputs.shape[1]

    boxes = []
    scores = []
    class_ids = []

    for i in range(rows):
        classes_scores = outputs[0][i][4:]
        (minScore, maxScore, minClassLoc, (x, maxClassIndex)) = cv2.minMaxLoc(classes_scores)
        if maxScore >= 0.25:
            box = [
                outputs[0][i][0] - (0.5 * outputs[0][i][2]), outputs[0][i][1] - (0.5 * outputs[0][i][3]),
                outputs[0][i][2], outputs[0][i][3]]
            boxes.append(box)
            scores.append(maxScore)
            class_ids.append(maxClassIndex)

    result_boxes = cv2.dnn.NMSBoxes(boxes, scores, 0.25, 0.45, 0.5)

    detections = []
    for i in range(len(result_boxes)):
        index = result_boxes[i]
        box = boxes[index]
        detection = {
            'class_id': class_ids[index],
            'class_name': CLASSES[class_ids[index]],
            'confidence': scores[index],
            'box': box,
            'scale': scale}
        detections.append(detection)
        draw_bounding_box(original_image, class_ids[index], scores[index], round(box[0] * scale), round(box[1] * scale),
                          round((box[0] + box[2]) * scale), round((box[1] + box[3]) * scale))

# Initialize the camera
url = "http://10.129.137.156:9000/mjpg"
camera = cv2.VideoCapture(url)

# Set the codec to MJPG if it is supported
if camera.isOpened():
    # camera.set(cv2.CAP_PROP_FRAME_WIDTH, 1280.0)
    # camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 720.0)
    camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc('M', 'J', 'P', 'G'))
else:
    raise IOError("Cannot open the webcam")

    # Define the codec and create VideoWriter object
# Get the width and height of the frames
frame_width = int(camera.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Frame width: {frame_width}, Frame height: {frame_height}")

# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi', fourcc, 30.0, (frame_width, frame_height))  # 20.0 is the frame rate

try:
    _start_time = time()
    while time() - _start_time >= 0:
        # Capture frame-by-frame
        ret, frame = camera.read()
        if not ret:
            raise IOError("Cannot capture frame")
        main(frame)
        out.write(frame)

        # Display the image
        clear_output(wait=True)
        
        # Afficher l'image capturée
        display(Image(data=cv2.imencode('.jpg', frame)[1]))

finally:
    # When everything done, release the capture
    camera.release()
    out.release()

8. 远程小车手动控制程序
- cpu模式：python remote_control.py
- npu模式：python remote_control.py --npu --om-model yolov8n.om